# Measure It · train on Colab
Select a GPU runtime and run these cells in order after the dataset and masks have been reviewed and published. The model and evaluation code live in the repository. A rerun requires a fresh run directory. Evaluate the test set only after checkpoint selection is frozen.

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
from google.colab import files

root = Path("/content/measure-it-envelopes")
if not root.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "dev-test01",
                    "https://github.com/Sherry-27/measure-it-envelopes.git", str(root)], check=True)
os.chdir(root)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Project revision", commit)
for split in ("train", "val", "test"):
    names = (root/"dataset"/"splits"/f"{split}.txt").read_text().splitlines()
    for name in names:
        if not (root/"dataset"/"images"/name).is_file() or not (root/"dataset"/"masks"/Path(name).with_suffix(".png")).is_file():
            raise FileNotFoundError(f"Add the real undistorted image and reviewed mask for {name} before training.")

In [ ]:
env = Path("/content/measure-it-venv")
if not env.exists():
    subprocess.run([sys.executable, "-m", "venv", str(env)], check=True)
python = str(env/"bin"/"python")
def run(*args):
    return subprocess.run([python, *args], cwd=root, check=True)
run("-m", "pip", "install", "-r", "requirements.txt")
run("-c", "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))")

In [ ]:
run("-m", "models.train", "--device", "cuda")
bundle = root/"models"/"runs"/"first"
(bundle/"git-revision.txt").write_text(commit+"\n")
(bundle/"pip-freeze.txt").write_text(subprocess.check_output([python, "-m", "pip", "freeze"], text=True))
(bundle/"gpu.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
(bundle/"machine.txt").write_text(subprocess.check_output(["lscpu"], text=True) + "\n" + Path("/proc/meminfo").read_text() + "\n" + Path("/etc/os-release").read_text())

In [ ]:
run("-m", "models.evaluate", "--device", "cuda", "--weights", str(bundle/"best.pth"), "--out", str(bundle/"evidence"))
import hashlib
with (bundle/"best.pth").open("rb") as f:
    (bundle/"sha256.txt").write_text(hashlib.file_digest(f, "sha256").hexdigest()+"  best.pth\n")
archive = shutil.make_archive("/content/envelope-training-evidence", "zip", bundle)
files.download(archive)